## Processed Dataset

In [1]:
import pandas as pd

df = pd.read_csv("../data/processed/shuffled_10_data.csv")

print(df.shape)
print(df.columns.tolist())
df.head()

(52800, 7)
['AC', 'PMID', 'Title', 'Abstract', 'Terms', 'Text_combined', 'batch_number']


,AC,PMID,Title,Abstract,Terms,Text_combined,batch_number
0,P06169,2185016,Autoregulation may control the expression of y...,Recently we deleted the pyruvate decarboxylase...,autoregulation,Autoregulation may control the expression of y...,1
1,P0AEM5,12704152,Complete genome sequence and comparative genom...,We determined the complete genome sequence of ...,NaN,Complete genome sequence and comparative genom...,1
2,B8FZE0,22316246,Genome sequence of Desulfitobacterium hafniens...,"The genome of the Gram-positive, metal-reducin...",NaN,Genome sequence of Desulfitobacterium hafniens...,1
3,P14656,12060286,Overlapping expression of cytosolic glutamine ...,In order to estimate whether cytosolic glutami...,NaN,Overlapping expression of cytosolic glutamine ...,1
4,Q7XXS4,27052628,Both overexpression and suppression of an Oryz...,Tight and accurate regulation of immunity and ...,autoactivation,Both overexpression and suppression of an Oryz...,1


## bert-base-uncased

In [2]:
# Imports
import os
import pandas as pd
import numpy as np
import re
import nltk
from nltk.corpus import stopwords
from sklearn.preprocessing import MultiLabelBinarizer

import torch
from torch import nn
from torch.optim.lr_scheduler import ReduceLROnPlateau
from transformers import AutoTokenizer, AutoModel
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import classification_report
from sklearn.metrics import f1_score, precision_score, recall_score
import matplotlib.pyplot as plt

In [3]:
# clean text
nltk.download('stopwords')
stop_words = set(stopwords.words('english'))

def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^\w\s]', '', text)
    text = re.sub(r'\d+', '', text)
    text = " ".join([word.strip() for word in text.split() if word not in stop_words])
    return text

df['Text_Cleaned'] = df['Text_combined'].apply(clean_text)

# fill nan with 'non-autoregulatory'
df['Terms'] = df['Terms'].fillna('non-autoregulatory')

# keep only selected columns
columns_to_keep = ['batch_number','Text_Cleaned','Terms']
df_cleaned = df[columns_to_keep]

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\35159\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [4]:
print(df_cleaned.shape)
df_cleaned.head()

(52800, 3)


,batch_number,Text_Cleaned,Terms
0,1,autoregulation may control expression yeast py...,autoregulation
1,1,complete genome sequence comparative genomics ...,non-autoregulatory
2,1,genome sequence desulfitobacterium hafniense d...,non-autoregulatory
3,1,overlapping expression cytosolic glutamine syn...,non-autoregulatory
4,1,overexpression suppression oryza sativa nblrrl...,autoactivation


In [5]:
# convert terms to list
df_cleaned['Terms_List'] = df_cleaned['Terms'].apply(
    lambda x: [term.strip() for term in x.split(',')]
)
df_cleaned['Terms_List'] = df_cleaned['Terms_List'].apply(lambda x: list(set(x)))

mlb = MultiLabelBinarizer()
labels = mlb.fit_transform(df_cleaned['Terms_List'])
label_columns = mlb.classes_

labels_df = pd.DataFrame(labels, columns=label_columns)
existing_columns = [col for col in label_columns if col in df_cleaned.columns]
df_cleaned = df_cleaned.drop(columns=existing_columns, errors='ignore')
df_cleaned = pd.concat([df_cleaned, labels_df], axis=1)

C:\Users\35159\AppData\Local\Temp\ipykernel_2368\626586514.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_cleaned['Terms_List'] = df_cleaned['Terms'].apply(
C:\Users\35159\AppData\Local\Temp\ipykernel_2368\626586514.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_cleaned['Terms_List'] = df_cleaned['Terms_List'].apply(lambda x: list(set(x)))


In [6]:
print(df_cleaned.shape)
df_cleaned.head()

(52800, 19)


,batch_number,Text_Cleaned,Terms,Terms_List,autoactivation,autocatalysis,autocatalytic,autofeedback,autoinducer,autoinduction,autoinhibition,autoinhibitory,autokinase,autolysis,autophosphorylation,autoregulation,autoregulatory,autoubiquitination,non-autoregulatory
0,1,autoregulation may control expression yeast py...,autoregulation,[autoregulation],0,0,0,0,0,0,0,0,0,0,0,1,0,0,0
1,1,complete genome sequence comparative genomics ...,non-autoregulatory,[non-autoregulatory],0,0,0,0,0,0,0,0,0,0,0,0,0,0,1
2,1,genome sequence desulfitobacterium hafniense d...,non-autoregulatory,[non-autoregulatory],0,0,0,0,0,0,0,0,0,0,0,0,0,0,1
3,1,overlapping expression cytosolic glutamine syn...,non-autoregulatory,[non-autoregulatory],0,0,0,0,0,0,0,0,0,0,0,0,0,0,1
4,1,overexpression suppression oryza sativa nblrrl...,autoactivation,[autoactivation],1,0,0,0,0,0,0,0,0,0,0,0,0,0,0


In [7]:
# Imports
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
from transformers import BertTokenizer, BertModel
from tqdm import tqdm

In [8]:
# Device and Tokenizer
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

In [9]:
# Load cleaned dataset
df = df_cleaned.copy()

In [10]:
# Define target labels (exclude metadata and non-autoregulatory)
label_cols = [col for col in df.columns if col not in [
    'batch_number', 'Text_Cleaned', 'Terms', 'Terms_List']]

In [11]:
# Dataset class
class BertMultiLabelDataset(Dataset):
    def __init__(self, dataframe, tokenizer, max_len=512):
        self.texts = dataframe["Text_Cleaned"].tolist()
        self.labels = dataframe[label_cols].values
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]
        labels = torch.tensor(self.labels[idx], dtype=torch.float32)
        encoding = self.tokenizer(
            text,
            truncation=True,
            padding="max_length",
            max_length=self.max_len,
            return_tensors="pt"
        )
        return {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "labels": labels
        }

In [12]:
# Model
class BertMultiLabelClassifier(nn.Module):
    def __init__(self, num_labels):
        super().__init__()
        self.bert = BertModel.from_pretrained("bert-base-uncased")
        self.dropout = nn.Dropout(0.3)
        self.classifier = nn.Linear(self.bert.config.hidden_size, num_labels)

    def forward(self, input_ids, attention_mask):
        output = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        pooled = output.pooler_output
        x = self.dropout(pooled)
        return self.classifier(x)

In [13]:
# Compute imbalance-aware weights
def compute_pos_weights(df_subset):
    weights = []
    for col in label_cols:
        pos = (df_subset[col] == 1).sum()
        neg = (df_subset[col] == 0).sum()
        weights.append(neg / pos if pos > 0 else 1.0)
    return torch.tensor(weights, dtype=torch.float32).to(device)

In [14]:
# Train/Eval functions
def train_one_epoch(model, dataloader, optimizer, criterion):
    model.train()
    total_loss = 0
    for batch in tqdm(dataloader, desc="Training"):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        logits = model(input_ids, attention_mask)
        loss = criterion(logits, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(dataloader)

def evaluate_model(model, dataloader):
    model.eval()
    preds, targets = [], []
    with torch.no_grad():
        for batch in dataloader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].cpu().numpy().astype(int)
            logits = model(input_ids, attention_mask)
            probs = torch.sigmoid(logits).cpu().numpy()
            preds.append((probs > 0.5).astype(int))
            targets.append(labels)
    preds = np.vstack(preds)
    targets = np.vstack(targets)
    return f1_score(targets, preds, average="micro", zero_division=0)

In [15]:
# Batch 1 Training
print("\n=== Training on Batch 1 ===")
df1 = df[df["batch_number"] == 1].copy()
train_df1, val_df1 = train_test_split(df1, test_size=0.2, random_state=42)
pos_weights_1 = compute_pos_weights(train_df1)
train_dl1 = DataLoader(BertMultiLabelDataset(train_df1, tokenizer), batch_size=16, shuffle=True)
val_dl1 = DataLoader(BertMultiLabelDataset(val_df1, tokenizer), batch_size=16)
model1 = BertMultiLabelClassifier(num_labels=len(label_cols)).to(device)
optimizer1 = optim.Adam(model1.parameters(), lr=2e-5)
criterion1 = nn.BCEWithLogitsLoss(pos_weight=pos_weights_1)

for epoch in range(3):
    loss = train_one_epoch(model1, train_dl1, optimizer1, criterion1)
    f1 = evaluate_model(model1, val_dl1)
    print(f"[Batch 1] Epoch {epoch+1} | Loss: {loss:.4f} | Val F1: {f1:.4f}")

# Batch 2 Training
print("\n=== Training on Batch 2 ===")
df2 = df[df["batch_number"] == 2].copy()
train_df2, val_df2 = train_test_split(df2, test_size=0.2, random_state=42)
pos_weights_2 = compute_pos_weights(train_df2)
train_dl2 = DataLoader(BertMultiLabelDataset(train_df2, tokenizer), batch_size=16, shuffle=True)
val_dl2 = DataLoader(BertMultiLabelDataset(val_df2, tokenizer), batch_size=16)
model2 = BertMultiLabelClassifier(num_labels=len(label_cols)).to(device)
optimizer2 = optim.Adam(model2.parameters(), lr=2e-5)
criterion2 = nn.BCEWithLogitsLoss(pos_weight=pos_weights_2)

for epoch in range(3):
    loss = train_one_epoch(model2, train_dl2, optimizer2, criterion2)
    f1 = evaluate_model(model2, val_dl2)
    print(f"[Batch 2] Epoch {epoch+1} | Loss: {loss:.4f} | Val F1: {f1:.4f}")


=== Training on Batch 1 ===


Training:   0%|          | 0/264 [00:00<?, ?it/s]c:\Users\35159\miniforge3\envs\autoregulatorycuda\lib\site-packages\transformers\models\bert\modeling_bert.py:440: UserWarning: 1Torch was not compiled with flash attention. (Triggered internally at C:\cb\pytorch_1000000000000\work\aten\src\ATen\native\transformers\cuda\sdp_utils.cpp:263.)
  attn_output = torch.nn.functional.scaled_dot_product_attention(
Training: 100%|██████████| 264/264 [19:13<00:00,  4.37s/it]


[Batch 1] Epoch 1 | Loss: 1.3225 | Val F1: 0.1933


Training: 100%|██████████| 264/264 [20:08<00:00,  4.58s/it]


[Batch 1] Epoch 2 | Loss: 1.2747 | Val F1: 0.1503


...

[Batch 2] Epoch 1 | Loss: 1.3138 | Val F1: 0.1094


Training: 100%|██████████| 264/264 [54:54<00:00, 12.48s/it]


[Batch 2] Epoch 2 | Loss: 1.3024 | Val F1: 0.2411


Training: 100%|██████████| 264/264 [54:50<00:00, 12.46s/it] 


[Batch 2] Epoch 3 | Loss: 0.9500 | Val F1: 0.4264


In [16]:
# Predict on test_data.csv
print("\n=== Predicting on Unseen Data ===")

# Load and clean test data
test_df = pd.read_csv("../data/processed/test_data.csv")

# If Text_Cleaned doesn't exist, generate it from Text_combined
if "Text_Cleaned" not in test_df.columns:
    def clean_text(text):
        text = text.lower()
        text = re.sub(r"[^\w\s]", "", text)
        text = re.sub(r"\d+", "", text)
        text = " ".join([word.strip() for word in text.split() if word not in stop_words])
        return text
    test_df["Text_Cleaned"] = test_df["Text_combined"].apply(clean_text)

# Prepare test dataset
test_texts = test_df["Text_Cleaned"].tolist()
test_ds = BertMultilabelDataset(test_df, tokenizer)
test_dl = DataLoader(test_ds, batch_size=1)

# Predict function
def predict_labels(model, dataloader):
    model.eval()
    preds = []
    with torch.no_grad():
        for batch in dataloader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            logits = model(input_ids, attention_mask)
            probs = torch.sigmoid(logits).cpu().numpy()
            pred_labels = (probs > 0.5).astype(int)
            preds.append(pred_labels[0])
    return np.array(preds)

# Make predictions
preds1 = predict_labels(model1, test_dl)
preds2 = predict_labels(model2, test_dl)

# Print comparison
print("\n=== Predictions on Test Samples ===\n")
for i, text in enumerate(test_texts):
    print(f"\n[Sample {i+1}]")
    print(f"Text: {text[:200]}...")
    for j, label in enumerate(label_cols):
        print(f"{label:<22} | Batch 1 → {preds1[i, j]}     Batch 2 → {preds2[i, j]}")


=== Predicting on Unseen Data ===


NameError: name 'BertMultilabelDataset' is not defined